In [16]:
from dotenv import load_dotenv

load_dotenv()

True

Create Subagents

In [17]:
from langchain.tools import tool
import random

@tool
def getKYCRisk(name: str) -> float:
    """Get the KYC risk for the Customer based on a provided name.
    Args:
        name (str): The full name of the user to look up.
    """
    # 1. Generate a random integer between 1 and 100
    num_int = random.randint(1, 100) # Comment: Both 1 and 100 are possible results

    # 2. Generate a random float between 0.0 and 1.0
    num_float = random.random()
    print(f"Get the KYC Risk for {name} is {num_float} ")
    return num_float

@tool
def getAMLRisk(name: str) -> float:
    """Get the AML risk for the Customer based on a provided name.
    Args:
        name (str): The full name of the user to look up.
    """
    # 1. Generate a random integer between 1 and 100
    num_int = random.randint(1, 100) # Comment: Both 1 and 100 are possible results

    # 2. Generate a random float between 0.0 and 1.0
    num_float = random.random()
    print(f"Get the AML Risk for {name} is {num_float}")
    return num_float

In [18]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[getKYCRisk]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[getAMLRisk]
)

In [19]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(name: str) -> float:
    """Call subagent 1 to get the  KYC risk for the name"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Get the KYC risk for {name}. Donot ask further questions respond only.")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(name: str) -> float:
    """Call subagent 2 to get the  KYC risk for the name"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Get the AML risk for {name}.Donot ask further questions respond only.")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="""
    You are a helpful assistant who can call subagents to calculate the final risk of the name. 
    Pass the name to both the sug agents.
    Final Risk = (KYC Risk + AML Risk )/2
    Output format : Risk of name is Final Risk
    """)

In [26]:
question = "What is the Final Risk of Jain Jose?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

Get the AML Risk for Jain Jose is 0.8096077557948747
Get the KYC Risk for Jain Jose is 0.11236712975087793 


In [21]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the Final Risk of Jain Jose?', additional_kwargs={}, response_metadata={}, id='9259d235-623e-4f2d-95c1-824de10cca94'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 447, 'prompt_tokens': 239, 'total_tokens': 686, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DaEIOPUzIympZjJxpPGnK8J4AO87e', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddcdf-8ad8-7e60-9d75-8d72be5b9568-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'name': 'Jain Jose'}, 'id': 'call_uX3mzY04XQZ9VAhb2Ec8h97P', 'type': 'tool_call'}, {'name': 'call_subagent_2', 'args': {'name': 'J

In [25]:
print(response["messages"][-1].content)

Risk of Jain Jose is 0.9377406724461987
